# Phase 4 — Parking Allocation

**BACKEND_PLAN.md §4 validation notebook.**

Exercises the three-function API in `agent/tools/parking.py` deterministically (no LLM, no MCP):

| Section | What it shows |
|---------|---------------|
| 1 | `estimate_apartments` — dwelling count from GFA |
| 2 | `parking_demand` — stall count + area budget |
| 3 | `compute_building_demand` — demand table for two buildings |
| 4 | `allocate_parking_zones` — basic allocation on a 90×60 site |
| 5 | Visualization — site plan with buildings + parking zones |
| 6 | Main-road preference — parking fronts the main road |
| 7 | Building-blocked south strip — parking relocates to another edge |
| 8 | Shortfall scenario — demand exceeds site capacity |
| 9 | Backend HTTP API check (SKIP if server is offline) |


In [ ]:
from __future__ import annotations
import sys
from pathlib import Path

workspace_root = Path.cwd().resolve()
candidate_roots = (
    workspace_root,
    workspace_root.parent,
    workspace_root / 'team_04',
    workspace_root.parent / 'team_04',
)
TEAM_ROOT = next((p for p in candidate_roots if (p / 'agent').exists()), None)
if TEAM_ROOT is None:
    raise FileNotFoundError('Run from workspace root, team_04/, or team_04/test_notebooks/')
if str(TEAM_ROOT) not in sys.path:
    sys.path.insert(0, str(TEAM_ROOT))

print('TEAM_ROOT:', TEAM_ROOT)

In [ ]:
import plotly.graph_objects as go

from agent.tools.parking import (
    BAY_DEPTH_M, M2_PER_STALL, STALL_WIDTH_M,
    allocate_parking_zones,
    compute_building_demand,
    estimate_apartments,
    parking_demand,
)

print('Parking tool imported OK')
print(f'  BAY_DEPTH_M={BAY_DEPTH_M} m  STALL_WIDTH_M={STALL_WIDTH_M} m  M2_PER_STALL={M2_PER_STALL} m²')

## Site and building definitions

- **Site**: 90 m × 60 m rectangle  
- **Main road**: along the south edge (width 20 m)  
- **Building A**: 5-storey L-shape in the south-west quadrant  
- **Building B**: 3-storey bar in the north-east quadrant

In [ ]:
SITE_BOUNDARY = [
    [0.0,  0.0,  0.0],
    [90.0, 0.0,  0.0],
    [90.0, 60.0, 0.0],
    [0.0,  60.0, 0.0],
    [0.0,  0.0,  0.0],
]

SITE_MODEL = {
    'boundary': SITE_BOUNDARY,
    'sides': [
        {'side_index': 0, 'start': [0.0,  0.0],  'end': [90.0, 0.0]},   # south — main road
        {'side_index': 1, 'start': [90.0, 0.0],  'end': [90.0, 60.0]},  # east
        {'side_index': 2, 'start': [90.0, 60.0], 'end': [0.0,  60.0]},  # north
        {'side_index': 3, 'start': [0.0,  60.0], 'end': [0.0,  0.0]},   # west
    ],
    'roads': {
        'main_road_side_index': 0,
        'main_road': {'name': 'Main Street', 'width_m': 20.0},
    },
}

BUILDING_A = {
    'building_id': 'bld_A',
    'label': 'Building A',
    'storeys': 5,
    'boundary': [
        [5.0,  16.0, 0.0],
        [40.0, 16.0, 0.0],
        [40.0, 30.0, 0.0],
        [18.0, 30.0, 0.0],
        [18.0, 45.0, 0.0],
        [5.0,  45.0, 0.0],
        [5.0,  16.0, 0.0],
    ],
}

BUILDING_B = {
    'building_id': 'bld_B',
    'label': 'Building B',
    'storeys': 3,
    'boundary': [
        [55.0, 25.0, 0.0],
        [85.0, 25.0, 0.0],
        [85.0, 50.0, 0.0],
        [55.0, 50.0, 0.0],
        [55.0, 25.0, 0.0],
    ],
}

BUILDINGS = [BUILDING_A, BUILDING_B]
print('Site area    :', 90 * 60, 'm²')
print('Buildings    :', len(BUILDINGS))

## §1 — estimate_apartments

In [ ]:
cases = [
    ('Small bar 200 m² × 3F', 200, 3),
    ('Medium L-shape 650 m² × 5F', 650, 5),
    ('Large H-shape 900 m² × 8F', 900, 8),
]
print(f'{'Building':35} {'Footprint':>10} {'Storeys':>7} {'GFA':>8} {'Net GFA':>9} {'Apts':>5}')
print('-' * 80)
for label, fp, st in cases:
    gfa = fp * st
    net = gfa * 0.80
    apts = estimate_apartments(fp, st)
    print(f'{label:35} {fp:>10.0f} {st:>7} {gfa:>8.0f} {net:>9.0f} {apts:>5}')

## §2 — parking_demand

In [ ]:
for apts in [0, 10, 25, 51]:
    d = parking_demand(apts)
    print(f'{apts:>3} apts → {d["stalls_required"]:>3} stalls, {d["area_sqm_required"]:>7.1f} m²  '
          f'(stall {d["stall_width_m"]}×{d["stall_depth_m"]} m + aisle {d["aisle_width_m"]} m)')

## §3 — compute_building_demand

In [ ]:
demand_table = compute_building_demand(BUILDINGS)
total_stalls = sum(d['stalls_required'] for d in demand_table)
total_area   = sum(d['area_sqm_required'] for d in demand_table)

print(f'{'Building ID':12} {'Apts':>5} {'Stalls':>6} {'Area (m²)':>10}')
print('-' * 40)
for d in demand_table:
    print(f'{d["building_id"]:12} {d["apartments"]:>5} {d["stalls_required"]:>6} {d["area_sqm_required"]:>10.1f}')
print('-' * 40)
print(f'{"TOTAL":12} {sum(d["apartments"] for d in demand_table):>5} {total_stalls:>6} {total_area:>10.1f}')

## §4 — allocate_parking_zones (basic)

In [ ]:
result = allocate_parking_zones(SITE_MODEL, BUILDINGS, demand_table)

print('=== ALLOCATION RESULT ===')
print(f'Stalls required  : {result["stalls_required"]}')
print(f'Stalls allocated : {result["total_stalls_allocated"]}')
print(f'Shortfall        : {result["shortfall"]}')
print(f'Feasible         : {result["feasible"]}')
print(f'Summary          : {result["summary"]}')
print()
print(f'{'Zone':18} {'Side':>5} {'Main Rd':>7} {'Stalls':>6} {'Area (m²)':>10}')
print('-' * 55)
for z in result['zones']:
    road_flag = '★' if z['is_main_road_side'] else ''
    print(f'{z["zone_id"]:18} {z["side_index"]:>5} {road_flag:>7} {z["stalls_allocated"]:>6} {z["area_sqm"]:>10.1f}')

## §5 — Site plan visualization

In [ ]:
def _xy(pts):
    xs = [p[0] for p in pts] + [pts[0][0]]
    ys = [p[1] for p in pts] + [pts[0][1]]
    return xs, ys

def _add_poly(fig, pts, name, fill='none', line_color='#333', line_width=2,
              dash=None, opacity=0.6, text=None):
    xs, ys = _xy(pts)
    fig.add_trace(go.Scatter(
        x=xs, y=ys, name=name, mode='lines+text' if text else 'lines',
        fill='toself' if fill != 'none' else None,
        fillcolor=fill if fill != 'none' else None,
        fillpattern=None,
        opacity=opacity,
        line=dict(color=line_color, width=line_width, dash=dash),
        text=[text] + [''] * (len(xs) - 1) if text else None,
        textposition='middle center',
        showlegend=name not in ('', None),
        hoverinfo='skip',
    ))

def make_site_plan(site_boundary, buildings, result, *, title='Parking Allocation'):
    fig = go.Figure()
    _add_poly(fig, site_boundary, 'Site boundary', line_color='#1d4ed8', line_width=3)

    bld_colors = ['rgba(15,118,110,0.45)', 'rgba(124,58,237,0.45)']
    for i, b in enumerate(buildings):
        cx = sum(p[0] for p in b['boundary']) / len(b['boundary'])
        cy = sum(p[1] for p in b['boundary']) / len(b['boundary'])
        xs, ys = _xy(b['boundary'])
        fig.add_trace(go.Scatter(
            x=xs, y=ys, name=b.get('label', f'Building {i+1}'),
            mode='lines', fill='toself',
            fillcolor=bld_colors[i % len(bld_colors)],
            line=dict(color=bld_colors[i % len(bld_colors)].replace('0.45', '1'), width=2),
            showlegend=True, hoverinfo='skip',
        ))
        fig.add_annotation(x=cx, y=cy, text=b.get('label', ''), showarrow=False,
                           font=dict(size=11, color='white', family='Arial Black'))

    for z in result.get('zones', []):
        cx = sum(p[0] for p in z['boundary']) / len(z['boundary'])
        cy = sum(p[1] for p in z['boundary']) / len(z['boundary'])
        xs, ys = _xy(z['boundary'])
        road_label = ' ★' if z['is_main_road_side'] else ''
        zone_name = f"{z['zone_id']}{road_label} ({z['stalls_allocated']} stalls)"
        fig.add_trace(go.Scatter(
            x=xs, y=ys, name=zone_name,
            mode='lines', fill='toself',
            fillcolor='rgba(234,179,8,0.30)',
            line=dict(color='#b45309', width=2, dash='dash'),
            showlegend=True, hoverinfo='skip',
        ))
        fig.add_annotation(
            x=cx, y=cy,
            text=f'P {z["stalls_allocated"]}',
            showarrow=False,
            font=dict(size=10, color='#7c2d12'),
        )

    fig.update_layout(
        title=title,
        yaxis=dict(scaleanchor='x', scaleratio=1, visible=False),
        xaxis=dict(visible=False),
        margin=dict(l=0, r=0, t=40, b=0),
        plot_bgcolor='#f0f4ff',
        paper_bgcolor='#f0f4ff',
        legend=dict(x=1.02, y=1, bgcolor='white', bordercolor='#ccc', borderwidth=1),
    )
    return fig

print('Helper functions defined')

In [ ]:
fig = make_site_plan(
    SITE_BOUNDARY, BUILDINGS, result,
    title=(
        f'Phase 4 Parking — {result["total_stalls_allocated"]}/{result["stalls_required"]} stalls '
        f'({len(result["zones"])} zone(s)) | Shortfall: {result["shortfall"]}'
    ),
)
fig.show()

## §6 — Main-road preference

When `prefer_road_side=True` (the default), the first parking zone fronts the main road (side 0 = south).  
When `prefer_road_side=False`, allocation goes to the longest side first.

In [ ]:
demand_5 = [{'building_id': 'bld_A', 'stalls_required': 5}]

r_road  = allocate_parking_zones(SITE_MODEL, [], demand_5, prefer_road_side=True)
r_noroad = allocate_parking_zones(SITE_MODEL, [], demand_5, prefer_road_side=False)

print('prefer_road_side=True:')
for z in r_road['zones']:
    print(f'  {z["zone_id"]}  side={z["side_index"]}  main_road={z["is_main_road_side"]}  stalls={z["stalls_allocated"]}')

print()
print('prefer_road_side=False:')
for z in r_noroad['zones']:
    print(f'  {z["zone_id"]}  side={z["side_index"]}  main_road={z["is_main_road_side"]}  stalls={z["stalls_allocated"]}')

assert r_road['zones'] and r_road['zones'][0]['is_main_road_side'], 'First zone should be main-road side'
print()
print('✓ First zone is correctly placed on the main-road side when prefer_road_side=True')

## §7 — Building blocks the south strip → parking relocates

Building C occupies the full south setback strip.  
The allocator must skip that edge and place parking on the next available edge.

In [ ]:
BUILDING_SOUTH_BLOCKER = {
    'building_id': 'bld_blocker',
    'label': 'Building C (south blocker)',
    'storeys': 4,
    'boundary': [
        [0.0,  0.0,  0.0],
        [90.0, 0.0,  0.0],
        [90.0, 14.0, 0.0],
        [0.0,  14.0, 0.0],
        [0.0,  0.0,  0.0],
    ],
}

r_blocked = allocate_parking_zones(
    SITE_MODEL, [BUILDING_SOUTH_BLOCKER],
    [{'building_id': 'bld_blocker', 'stalls_required': 8}],
)

print('Allocation with south strip blocked:')
print(f'  Stalls required  : {r_blocked["stalls_required"]}')
print(f'  Stalls allocated : {r_blocked["total_stalls_allocated"]}')
for z in r_blocked['zones']:
    print(f'  {z["zone_id"]}  side={z["side_index"]}  main_road={z["is_main_road_side"]}  stalls={z["stalls_allocated"]}')

# All zones must NOT overlap the south blocker
from shapely.geometry import Polygon as SP
blocker_poly = SP([(p[0], p[1]) for p in BUILDING_SOUTH_BLOCKER['boundary']])
for z in r_blocked['zones']:
    zp = SP([(p[0], p[1]) for p in z['boundary']])
    assert zp.intersection(blocker_poly).area < 0.5, f'Zone {z["zone_id"]} overlaps the blocker!'
print('\n✓ No parking zone overlaps the south-strip building')

fig_b = make_site_plan(
    SITE_BOUNDARY, [BUILDING_SOUTH_BLOCKER], r_blocked,
    title='South strip blocked — parking relocates to another edge',
)
fig_b.show()

## §8 — Shortfall scenario

A very small site (20 × 20 m) with high parking demand cannot fit all stalls.  
The allocator reports `shortfall > 0` and `feasible = False`.

In [ ]:
TINY_SITE_MODEL = {
    'boundary': [[0,0,0],[20,0,0],[20,20,0],[0,20,0],[0,0,0]],
    'sides': [
        {'side_index': 0, 'start': [0,0],  'end': [20,0]},
        {'side_index': 1, 'start': [20,0], 'end': [20,20]},
        {'side_index': 2, 'start': [20,20],'end': [0,20]},
        {'side_index': 3, 'start': [0,20], 'end': [0,0]},
    ],
}

r_shortfall = allocate_parking_zones(
    TINY_SITE_MODEL, [],
    [{'building_id': 'big_building', 'stalls_required': 200}],
)
print('Shortfall scenario (20×20 site, 200 stalls needed):')
print(f'  Stalls required  : {r_shortfall["stalls_required"]}')
print(f'  Stalls allocated : {r_shortfall["total_stalls_allocated"]}')
print(f'  Shortfall        : {r_shortfall["shortfall"]}')
print(f'  Feasible         : {r_shortfall["feasible"]}')
print(f'  Summary          : {r_shortfall["summary"]}')
assert r_shortfall['shortfall'] > 0
assert not r_shortfall['feasible']
print('\n✓ Shortfall correctly reported')

## §9 — Backend HTTP API check

Calls the live FastAPI server to confirm the Phase 4 backend routes are registered  
under `POST /tools/{estimate_apartments, parking_demand, building_demand, parking_allocation}`.

Start the server first if not running:
```
uvicorn team_04.backend.app:app --reload --port 8000
```

In [ ]:
import json as _json
import urllib.request, urllib.error

_BASE = 'http://localhost:8000'

def _post_tool(tool: str, args: dict) -> dict:
    body = _json.dumps({'tool_name': tool, 'arguments': args}).encode()
    req = urllib.request.Request(
        f'{_BASE}/tools/{tool}', data=body, method='POST',
        headers={'Content-Type': 'application/json'},
    )
    with urllib.request.urlopen(req, timeout=10) as r:
        return _json.loads(r.read())

api_checks = []

def _api(name, fn):
    try:
        api_checks.append((name, 'OK', str(fn())[:120]))
    except urllib.error.URLError as exc:
        api_checks.append((name, 'SKIP', f'server offline — {exc.reason}'))
    except Exception as exc:
        api_checks.append((name, 'FAIL', f'{type(exc).__name__}: {exc}'))

# 1. Verify tools appear in the registry
_api('GET /tools (parking tools present)', lambda: sorted(
    t for t in _json.loads(
        urllib.request.urlopen(f'{_BASE}/tools', timeout=5).read()
    )['tools']
    if 'parking' in t or 'apartment' in t or 'demand' in t
))

# 2. estimate_apartments
_api('POST /tools/estimate_apartments', lambda: (
    lambda r: f'success={r["success"]}  apartments={r["result"]}'
)(_post_tool('estimate_apartments', {'footprint_area_sqm': 650, 'storeys': 5})))

# 3. parking_demand
_api('POST /tools/parking_demand', lambda: (
    lambda r: f'success={r["success"]}  stalls={r["result"]["stalls_required"]}'
)(_post_tool('parking_demand', {'apartments': 37})))

# 4. building_demand (list of buildings)
_api('POST /tools/building_demand', lambda: (
    lambda r: f'success={r["success"]}  entries={len(r["result"])}'
)(_post_tool('building_demand', {'buildings': [
    {'building_id': 'a', 'boundary': BUILDING_A['boundary'], 'storeys': 5},
    {'building_id': 'b', 'boundary': BUILDING_B['boundary'], 'storeys': 3},
]})))

# 5. parking_allocation
_api('POST /tools/parking_allocation', lambda: (
    lambda r: (
        f'success={r["success"]}  '
        f'stalls_allocated={r["result"]["total_stalls_allocated"]}  '
        f'zones={len(r["result"]["zones"])}'
    )
)(_post_tool('parking_allocation', {
    'site_model': SITE_MODEL,
    'buildings': [BUILDING_A, BUILDING_B],
    'demand_by_building': demand_table,
})))

print(f'{"ENDPOINT":<52} {"STATUS":6} DETAIL')
print('-' * 110)
for name, status, detail in api_checks:
    print(f'{name:<52} {status:6} {detail}')

n_ok   = sum(1 for _,s,_ in api_checks if s == 'OK')
n_skip = sum(1 for _,s,_ in api_checks if s == 'SKIP')
n_fail = sum(1 for _,s,_ in api_checks if s == 'FAIL')
print(f'\n{n_ok} OK  {n_skip} SKIP (server offline)  {n_fail} FAIL')
if n_fail:
    raise AssertionError('Phase 4 backend API checks FAILED — see FAIL rows.')
if n_skip == len(api_checks):
    print('\nAll checks skipped. Start the backend to test:')
    print('  uvicorn team_04.backend.app:app --reload --port 8000')

## Summary

| Check | Result |
|-------|--------|
| `estimate_apartments` math | ✓ |
| `parking_demand` stall count + area | ✓ |
| `compute_building_demand` two buildings | ✓ |
| `allocate_parking_zones` basic allocation | ✓ |
| Main-road preference (first zone on south edge) | ✓ |
| Building blocks south → parking relocates | ✓ |
| Shortfall reported when site too small | ✓ |
| Backend API (SKIP if server offline) | checked above |

All geometry tools are pure functions (Shapely in / dict out).  
The 34-test regression suite is in `benchmarking/test_parking.py`.